# Feature engineering

This notebook fits all learned feature transformations on gold training data only. It writes one unified artifact per split containing every feature representation required by downstream models and explanations.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pyspark.ml.feature import (
    CountVectorizer, HashingTF, IDF, MinHashLSH, MinMaxScaler,
    StandardScaler, VectorAssembler, Word2Vec
)
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, StringType

SEED = 42
BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
AUDIT_DIR = BASE_DIR / "audit"
EDA_DIR = BASE_DIR / "eda"
for directory in (MODELS_DIR, AUDIT_DIR, EDA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_Features")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

datasets = {
    "train": spark.read.parquet(str(PROCESSED_DIR / "train_features.parquet")),
    "validation": spark.read.parquet(str(PROCESSED_DIR / "validation_features.parquet")),
    "test": spark.read.parquet(str(PROCESSED_DIR / "test_features.parquet")),
    "banglish": spark.read.parquet(str(PROCESSED_DIR / "banglish_features.parquet")),
}
train = datasets["train"]


In [ ]:
count_vectorizer = CountVectorizer(
    inputCol="tokens",
    outputCol="term_frequency",
    vocabSize=50000,
    minDF=3.0,
)
count_vectorizer_model = count_vectorizer.fit(train)
train_term_frequency = count_vectorizer_model.transform(train)

idf = IDF(
    inputCol="term_frequency",
    outputCol="tfidf",
    minDocFreq=3,
)
idf_model = idf.fit(train_term_frequency)

word2vec = Word2Vec(
    inputCol="tokens",
    outputCol="word2vec",
    vectorSize=100,
    minCount=3,
    maxIter=10,
    numPartitions=4,
    seed=SEED,
)
word2vec_model = word2vec.fit(train)

count_vectorizer_model.write().overwrite().save(
    str(MODELS_DIR / "count_vectorizer")
)
idf_model.write().overwrite().save(str(MODELS_DIR / "idf"))
word2vec_model.write().overwrite().save(str(MODELS_DIR / "word2vec"))


In [ ]:
@F.udf(ArrayType(StringType()))
def token_shingles(tokens):
    values = [str(token) for token in (tokens or []) if str(token)]
    if len(values) < 3:
        return ["__short__:" + "_".join(values)] if values else ["__empty__"]
    return [
        "_".join(values[index:index + 3])
        for index in range(len(values) - 2)
    ]


hashing = HashingTF(
    inputCol="token_shingles",
    outputCol="shingle_vector",
    numFeatures=16384,
    binary=True,
)

def add_shingle_vector(frame):
    return hashing.transform(
        frame.withColumn("token_shingles", token_shingles("tokens"))
    )

train_shingles = add_shingle_vector(train)
minhash_model = MinHashLSH(
    inputCol="shingle_vector",
    outputCol="minhash_signature",
    numHashTables=5,
    seed=SEED,
).fit(train_shingles)
minhash_model.write().overwrite().save(str(MODELS_DIR / "minhash"))

reference = minhash_model.transform(train_shingles).select(
    "review_id", "shingle_vector", "minhash_signature"
)

def add_near_duplicate_features(frame, self_comparison=False):
    query = minhash_model.transform(add_shingle_vector(frame)).select(
        "review_id", "shingle_vector", "minhash_signature"
    )
    pairs = minhash_model.approxSimilarityJoin(
        query,
        reference,
        threshold=0.30,
        distCol="jaccard_distance",
    )
    if self_comparison:
        pairs = pairs.filter(
            F.col("datasetA.review_id") != F.col("datasetB.review_id")
        )
    pair_summary = (
        pairs.select(
            F.col("datasetA.review_id").alias("review_id"),
            F.col("datasetB.review_id").alias("neighbor_id"),
            (1.0 - F.col("jaccard_distance")).alias("similarity"),
        )
        .dropDuplicates(["review_id", "neighbor_id"])
        .groupBy("review_id")
        .agg(
            F.count("*").cast("double").alias("near_duplicate_degree"),
            F.max("similarity").alias("near_duplicate_similarity"),
        )
    )
    return (
        frame.join(pair_summary, "review_id", "left")
        .fillna({
            "near_duplicate_degree": 0.0,
            "near_duplicate_similarity": 0.0,
        })
        .withColumn(
            "near_duplicate_cluster_size",
            F.col("near_duplicate_degree") + F.lit(1.0),
        )
    )

near_datasets = {
    name: add_near_duplicate_features(frame, self_comparison=(name == "train"))
    for name, frame in datasets.items()
}


In [ ]:
HANDCRAFTED_FEATURES = [
    "char_length", "token_count", "codemix_ratio", "repeated_char_ratio",
    "url_count", "email_count", "phone_count", "price_count",
    "hashtag_count", "mention_count", "exclamation_count", "emoji_count",
    "promo_keyword_hits", "bengali_chars", "latin_chars", "digit_chars",
    "other_chars", "exact_dup_cluster_size", "is_exact_duplicate",
    "near_duplicate_degree", "near_duplicate_similarity",
    "near_duplicate_cluster_size", "has_url", "has_email", "has_phone",
    "has_price", "has_promo", "has_emoji",
]
NO_NEAR_DUPLICATE_FEATURES = [
    feature for feature in HANDCRAFTED_FEATURES
    if not feature.startswith("near_duplicate_")
]

handcrafted_assembler = VectorAssembler(
    inputCols=HANDCRAFTED_FEATURES,
    outputCol="handcrafted_raw",
)
no_near_assembler = VectorAssembler(
    inputCols=NO_NEAR_DUPLICATE_FEATURES,
    outputCol="handcrafted_no_near_raw",
)

train_assembled = no_near_assembler.transform(
    handcrafted_assembler.transform(near_datasets["train"])
)
handcrafted_scaler_model = StandardScaler(
    inputCol="handcrafted_raw",
    outputCol="handcrafted_scaled",
    withMean=False,
    withStd=True,
).fit(train_assembled)
no_near_scaler_model = StandardScaler(
    inputCol="handcrafted_no_near_raw",
    outputCol="handcrafted_no_near_scaled",
    withMean=False,
    withStd=True,
).fit(train_assembled)
naive_bayes_scaler_model = MinMaxScaler(
    inputCol="handcrafted_raw",
    outputCol="handcrafted_nonnegative",
).fit(train_assembled)

handcrafted_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "handcrafted_scaler")
)
no_near_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "no_near_scaler")
)
naive_bayes_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "naive_bayes_scaler")
)

linear_assembler = VectorAssembler(
    inputCols=["tfidf", "handcrafted_scaled"],
    outputCol="features_linear",
)
linear_no_near_assembler = VectorAssembler(
    inputCols=["tfidf", "handcrafted_no_near_scaled"],
    outputCol="features_linear_no_near",
)
tree_assembler = VectorAssembler(
    inputCols=["word2vec", "handcrafted_raw"],
    outputCol="features_tree",
)
naive_bayes_assembler = VectorAssembler(
    inputCols=["term_frequency", "handcrafted_nonnegative"],
    outputCol="features_nb",
)

def build_feature_matrix(frame):
    transformed = count_vectorizer_model.transform(frame)
    transformed = idf_model.transform(transformed)
    transformed = word2vec_model.transform(transformed)
    transformed = handcrafted_assembler.transform(transformed)
    transformed = no_near_assembler.transform(transformed)
    transformed = handcrafted_scaler_model.transform(transformed)
    transformed = no_near_scaler_model.transform(transformed)
    transformed = naive_bayes_scaler_model.transform(transformed)
    transformed = linear_assembler.transform(transformed)
    transformed = linear_no_near_assembler.transform(transformed)
    transformed = tree_assembler.transform(transformed)
    transformed = naive_bayes_assembler.transform(transformed)
    return transformed.withColumn("features_tfidf_only", F.col("tfidf"))

model_ready = {
    name: build_feature_matrix(frame)
    for name, frame in near_datasets.items()
}

required_columns = {
    "review_id", "safe_review", "model_text", "tokens", "source_dataset",
    "source_split", "label_source", "features_linear", "features_tree",
    "features_nb", "features_tfidf_only", "features_linear_no_near",
    "handcrafted_raw", "handcrafted_scaled", "handcrafted_nonnegative",
}
for name, frame in model_ready.items():
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise AssertionError(f"{name} is missing columns: {missing}")
    frame.write.mode("overwrite").parquet(
        str(PROCESSED_DIR / f"{name}_model_ready.parquet")
    )


In [ ]:
near_duplicate_summary = []
for name, frame in model_ready.items():
    row = frame.agg(
        F.count("*").alias("rows"),
        F.avg("near_duplicate_degree").alias("mean_degree"),
        F.max("near_duplicate_degree").alias("maximum_degree"),
        F.avg("near_duplicate_similarity").alias("mean_similarity"),
    ).first()
    near_duplicate_summary.append({
        "dataset": name,
        "rows": int(row["rows"]),
        "mean_degree": float(row["mean_degree"] or 0.0),
        "maximum_degree": float(row["maximum_degree"] or 0.0),
        "mean_similarity": float(row["mean_similarity"] or 0.0),
    })

pd.DataFrame(near_duplicate_summary).to_csv(
    EDA_DIR / "near_duplicate_summary.csv",
    index=False,
)

sample_linear = model_ready["train"].select("features_linear").first()[0]
sample_tree = model_ready["train"].select("features_tree").first()[0]
sample_nb = model_ready["train"].select("features_nb").first()[0]

feature_audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "fit_scope": "BFRD training split only",
    "count_vectorizer_vocabulary": len(count_vectorizer_model.vocabulary),
    "word2vec_dimensions": 100,
    "minhash": {
        "token_shingle_size": 3,
        "distance_threshold": 0.30,
        "hash_tables": 5,
        "hash_dimensions": 16384,
    },
    "feature_dimensions": {
        "linear": sample_linear.size,
        "tree": sample_tree.size,
        "naive_bayes": sample_nb.size,
    },
    "handcrafted_features": HANDCRAFTED_FEATURES,
    "near_duplicate_summary": near_duplicate_summary,
}
with open(AUDIT_DIR / "feature_audit.json", "w", encoding="utf-8") as handle:
    json.dump(feature_audit, handle, ensure_ascii=False, indent=2)

print(json.dumps(feature_audit, ensure_ascii=False, indent=2))
